# LABORATORIO 6: EL ASISTENTE DE CARGA Y SNOWFLAKE MARKETPLACE

👉 En esta lección, aprenderemos sobre cargar archivos y los tipos de objetos de Snowflake que pueden facilitar ese proceso. También nos familiarizaremos con Snowflake Marketplace, que se asemeja a una tienda de aplicaciones para datos y productos de datos.

Para comenzar, reunamos la **información de contexto** que usaremos durante este laboratorio. 

- Haga clic en el botón **Start** para activar este cuaderno.

- Ejecute la siguiente celda Python.


#### :warning: Debe volver a ejecutar la celda a continuación siempre que se inicie una nueva sesión para este cuaderno para configurar “variables” para usar en celdas posteriores. :warning:


In [ ]:
import streamlit as st
from snowflake.snowpark.context import get_active_session
session = get_active_session()
user = session.get_current_user().strip('"')
your_db = user + '_DB'
print('Su información de contexto actual:')
print('---------------------------------')
print(session)
print('Su usuario actual es' + user)

## Las instrucciones de INSERT envejecen rápido 📓 

### Datos de la tabla vegetable_details

Supongamos que los analistas de negocios de nuestro equipo deciden acortar la columna Profundidad de raíz a una sola letra y generar un archivo de valores separados por comas (CSV). Este cambio está bien por sí solo, pero hay problemas en la preparación: 
- ¿Puede ver el problema? 
- ¿Por qué no puede cargar este CSV? 
- ¿Cuántas comas hay en la mayoría de las filas? 
- ¿Cuántas comas hay en la fila de pimientos rojos? 

![Datos de la tabla vegetable_details (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_veg_details_table_1.png)


### Crear una tabla vegetable_details 🥋 

Al trabajar en la celda de un cuaderno, recuerde que podemos especificar explícitamente nuestro contexto con una instrucción `USE` (por ej., `USE SCHEMA db_name/schema_name`).

Una alternativa a esto es usar una **notación de objetos completa**. Esto significa que usamos la ruta (o jerarquía) completa para un objeto siempre que hacemos referencia a un objeto en nuestra SQL. Es más detallado, pero puede ayudar en relación con la claridad. La estructura general de esta convención para la asignación de nombres sigue el formato: `<database_name>.<schema_name>.<object_name>`.

💡 **Sugerencia**: Si crea la tabla en la base de datos o esquema equivocado, puede usar una instrucción `ALTER TABLE - RENAME` para moverla, o puede arrastrar la tabla, actualizar la configuración contextual del menú desplegable y volver a crearla. 


In [ ]:
-- Cree una tabla Vegetable_Details

CREATE TABLE IF NOT EXISTS {{user}}_garden_plants.veggies.vegetable_details (
    plant_name VARCHAR(25),
    root_depth_code VARCHAR(1)
) 
DATA_RETENTION_TIME_IN_DAYS = 7;

## Cargar filas de tablas de un archivo 

### Derivar los datos 

Se han cargado dos archivos CSV (valores separados por comas) con los nombres `veggie_details_a_to_k_comma_opt_enclosed.csv` y `veggie_details_k_to_z_pipe.csv` en la fase **COMMON_DB.RESOURCES.CLASS_FILES**. Queremos cargar estos datos en la tabla **VEGETABLE_DETAILS** que creó. 

### Descargue el primer archivo 🥋 

Ejecute la celda del código Python siguiente y haga clic en el enlace generado para descargar un archivo (Artichoke a Kale).


In [ ]:
import streamlit as st
snowpark_df = session.sql("SELECT GET_PRESIGNED_URL(@common_db.resources.course_files, 'veggie_details_a_to_k_comma_opt_enclosed.csv')")
collected_data = snowpark_df.collect()
st.write('Haga clic en el siguiente enlace para descargar el archivo:')
st.write(collected_data[0][0])

### Pasos del cuadro de diálogo Load Data 🥋 

1. Localice la tabla **VEGETABLE_DETAILS** en el navegador de objetos de Snowsight.
1. Haga clic en el ícono **Open Table details in new tab** para abrir esta pantalla en una nueva pestaña.

![Cuadro de diálogo Load Data (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_open_table_details_1.png)

1. Haga clic en el botón `Load Data` azul arriba a la derecha.
1. Aparecerá el siguiente cuadro de diálogo.
1. Si se le indica que use un almacén, seleccione el almacén con el nombre que seleccionó: **(animal)_WH**.

![Cuadro de diálogo Load Data (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_load_data_1.png)

1. Haga clic en el botón azul **Browse**.
1. Localice el archivo que descargó localmente, con el nombre `veggie_details_a_to_k_comma_opt_enclosed.csv` y selecciónelo para abrir.
1. Luego, haga clic en el botón azul **Next** que se encuentra en la esquina inferior derecha del cuadro de diálogo.
1. Aparece un cuadro de diálogo extendido Load Data.
1. Aceptaremos muchos de los valores predeterminados elegidos para nuestro archivo, pero haga clic en la flecha desplegable para **View options** en el cuadro **File format** a la izquierda para ajustar algunas opciones.

![Cuadro de diálogo Extended Load Data (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_load_data_2.png)

1. Seleccione “Skip first line” para **Header**.
1. Seleccione “Double quotes (default)” para **Field optionally enclosed by**.
1. Haga clic en el botón azul **Load** que se encuentra en la esquina inferior derecha del cuadro de diálogo.

![Opciones de archivo de datos de carga (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_load_data_3.png)

1. Si todo va bien, debería ver al siguiente cuadro de diálogo de éxito.
1. Haga clic en **Done** y use la pestaña de opción **Data Preview** en la página que aparece para revisar los datos cargados en su tabla.

![Carga de datos exitosa (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_load_data_success_1_v2.png)


### Recuperarse de los errores cuando se cargan datos 📓

Si accidentalmente carga el mismo archivo dos veces y quiere comenzar otra vez, ejecute un comando `TRUNCATE` para vaciar la tabla. 

`TRUNCATE TABLE (animal)_GARDEN_PLANTS.VEGGIES.VEGETABLE_DETAILS;`

Luego, inicie el proceso de carga de nuevo. 


## Cargar filas de tablas de un archivo

### Descargue el segundo archivo 🥋 

Ejecute la celda del código Python siguiente y haga clic en el enlace generado para descargar nuestro segundo archivo (Kohlrabi a Zucchini).


In [ ]:
snowpark_df = session.sql("SELECT GET_PRESIGNED_URL(@common_db.resources.course_files, 'veggie_details_k_to_z_pipe.csv')")
collected_data = snowpark_df.collect()
st.write('Haga clic en el siguiente enlace para descargar el archivo:')
st.write(collected_data[0][0])

## Laboratorio de desafío: Cargue el archivo en la tabla 🎯 

Cargue el archivo que descargó en la misma tabla mediante el mismo método **Load Data** que usó. 

💡 **Nota**: Al menos una configuración en **View options** será diferente cuando cargue este archivo en comparación con el primero. Las columnas **NO** están separadas por comas esta vez. En muchos casos, Snowflake puede detectar de manera automática la forma de los archivos que se cargan y responderá con opciones para respaldar la carga correcta. ¿Sucede esto con la carga de su segundo archivo?

![Datos de la tabla de detalles de verduras (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_veg_details_csv_1.png)

💡 **CONSEJO**: No use Excel para abrir archivos CSV si quiere ver los caracteres que se usan para separar las columnas y las filas. Use un editor de texto simple. En Windows, el Bloc de notas funciona a la perfección. En Mac, puede usar TextEdit. 

Una vez cargado el archivo, compruebe su tabla ejecutando una instrucción `SELECT` y revisando el resultado. Debería ver 42 filas. Si revierte el orden de clasificación (haga clic en el encabezado de columna **PLANT_NAME** en la tabla de resultado). Debería poder ver Zucchini arriba, lo que le demuestra que cargó el segundo archivo.


In [ ]:
SELECT * 
FROM {{user}}_garden_plants.veggies.vegetable_details;

## Ver los datos de nuestra tabla 🎯 

### Ver la tabla vegetable_details mediante SQL 

Este laboratorio de desafío no incluye detalles paso por paso, solo pautas generales para lograr varios objetivos.

1. Vea sus datos.
1. Aísle los registros de “Espinaca”. (Spinach)
1. Observe que el nombre de una planta aparece dos veces en el conjunto de datos.
1. Encuentre una manera de eliminarlo.
1. Vea sus datos otra vez.

💡 **CONSEJO**: **Reemplace** los caracteres hash (`#`) en las consultas según complejidad a continuación. Cuando vea una, se requiere su entrada. 

💡 **CONSEJO**: Si realmente está atascado, **expanda** las celdas de pistas debajo de cada instrucción SQL “clasificada según la complejidad” haciendo clic en **Code and results** del selector **View display options** junto al botón de reproducción gris más a la derecha de la celda. No olvide probar por su cuenta primero :grinning:.


### 1\. Ver sus datos. 🎯 


In [ ]:
-- modifique el siguiente código y ejecútelo para ver todos los datos de la tabla después de establecer su contexto (base de datos y esquema)
USE DATABASE MALLARD04_GARDEN_PLANTS;
USE SCHEMA veggies;

SELECT *
FROM VEGETABLE_DETAILS;

In [ ]:
/* 
-- también podría codificar la siguiente instrucción USE para su usuario animal en lugar de usar la sustitución de variable
USE SCHEMA {{user}}_garden_plants.veggies;

SELECT *
FROM vegetable_details;
*/

### 2\. Escribir una consulta SQL para devolver todas las filas en las que **plant_name** es **Spinach**. 🎯 

Nosotros usaremos la función [`UPPER()`](https://docs.snowflake.com/en/sql-reference/functions/upper). Esto significa que no tenemos que preocuparnos por escribir la cadena de prueba que estamos comprobando como `spinach` o `Spinach`, o cualquier otra variación. Nuestra entrada y los valores almacenados en la columna se convertirán a `SPINACH`.

💡 **Sugerencia**: ¡Nos sorprendimos al ver dos filas Spinach!


In [ ]:
SELECT *
FROM VEGETABLE_DETAILS
WHERE UPPER(plant_name) = UPPER('SPINACH');

In [ ]:
/*
SELECT *
FROM vegetable_details
WHERE UPPER(plant_name) = UPPER('Spinach');
*/

### 3\. Una de las filas tiene una “S” que corresponde a raíces superficiales, y la otra tiene una “D” que corresponde a raíces profundas. 🎯 

Debemos eliminar la fila que dice que las raíces de la espinaca son profundas. Primero, aislemos la fila “D”. 


In [ ]:
SELECT *
FROM vegetable_details
WHERE UPPER(plant_name) = UPPER('SPINACH')
AND root_depth_code = 'D';

In [ ]:
/*
SELECT *
FROM vegetable_details
WHERE UPPER(plant_name) = UPPER('Spinach')
AND root_depth_code = 'D';
*/

### 4\. Quite solo la fila Espinaca con la letra “D” en la columna ROOT_DEPTH_CODE. 🎯 


In [ ]:
DELETE
FROM vegetable_details
WHERE UPPER(plant_name) = UPPER('SPINACH')
AND root_depth_code = 'D';

In [ ]:
/*
DELETE
FROM vegetable_details
WHERE UPPER(plant_name) = UPPER('Spinach')
AND root_depth_code = 'D';
*/

### 5\. Observe todos los datos de nuevo y asegúrese de que no hay nombres de verduras que aparezcan dos veces. 🎯 


In [ ]:
SELECT *
FROM vegetable_details;

In [ ]:
/*
SELECT *
FROM vegetable_details;
*/

### Una observación sobre el objeto File Format 📓

En su trabajo para este laboratorio, ha usado el asistente Carga de datos de Snowsight para guiar a Snowflake en la lectura de datos de un archivo y su inserción en una tabla. Como pudo ver, el cuadro de diálogo incluye opciones para ayudar a que Snowflake comprenda la “forma” de los datos como están en el archivo de origen y a especificar cómo se deben interpretar y procesar los datos en el archivo. Esto incluye el tipo (por ej., CSV), los delimitadores de campos y si incluye líneas de encabezado que deberían omitirse.

Estas instrucciones se pueden agrupar en un objeto de Snowflake llamado [FILE FORMAT](https://docs.snowflake.com/en/sql-reference/sql/create-file-format), al que se hace referencia en el cuadro de diálogo Load Data o cuando se cargan datos mediante programación en Snowflake a través del comando [`COPY INTO`](https://docs.snowflake.com/en/sql-reference/sql/copy-into-table). Esto le ahorra tiempo que no tiene para volver a ingresar o codificar estas instrucciones para repetidas cargas de archivos con las mismas especificaciones.


## Una breve introducción a la clonación 📓

La función [zero-copy cloning](https://docs.snowflake.com/en/user-guide/object-clone) de Snowflake proporciona una forma conveniente de tomar una “instantánea” de manera rápida de cualquier tabla, esquema o base de datos (y otros objetos) y crear una copia derivada de ese objeto, que inicialmente comparte el almacenamiento subyacente. Esto no implica copiar físicamente los datos subyacentes, sino que es una operación que solo afecta a los metadatos. Por este motivo, generalmente es muy rápido clonar tablas, esquemas o incluso bases de datos enteras.

Los clones son objetos independientes en el momento en que se crean. Puede realizar las mismas operaciones en los objetos clonados que realiza en los objetos de origen. Por ejemplo, puede hacer todo con una tabla que clonó, incluso escribir en ella o cambiar sus parámetros con el comando `ALTER TABLE`.

La clonación puede ser extremadamente útil para crear respaldos instantáneos sin costos adicionales (hasta que se hagan cambios en el objeto clonado). Esta función se usa a menudo en Snowflake para el aprovisionamiento veloz de entornos de desarrollo y prueba/control de calidad y respaldos de datos.


### Clonar una tabla 🥋

A continuación, deberá crear un nuevo esquema llamado **CLONED_OBJECTS** en la base de datos **(animal)_DB**. Luego, `CLONE` la tabla **VEGETABLE_DETAILS** existente en este nuevo esquema con el nombre **VEGETABLE_DETAILS_CLONE**.

Ejecute el siguiente código para realizar estos pasos y observe la velocidad de esta operación. Recuerde que al clonar en Snowflake no se copian los datos de la tabla, solo los metadatos (datos sobre los datos).


In [ ]:
CREATE SCHEMA IF NOT EXISTS {{user}}_db.cloned_objects;

USE SCHEMA {{user}}_db.cloned_objects;

CREATE TABLE IF NOT EXISTS vegetable_details_clone
    CLONE {{user}}_garden_plants.veggies.vegetable_details;

### Revise los detalles sobre la tabla clonada

Use el comando `SHOW` para revisar la tabla clonada junto con la tabla de origen desde la que se originó. Son objetos independientes. Sin embargo, observe que ambos comparten el mismo número de filas y bytes, si bien el almacenamiento solo se medirá en función de la tabla de origen.


In [ ]:
SHOW TABLES LIKE 'vegetable_details%' IN ACCOUNT;

### Consultar la tabla clonada

Consulte la tabla clonada **VEGETABLE_DETAILS_CLONE** como lo hicimos con la tabla de la que proviene, **VEGETABLE_DETAILS**. Estos datos deberían ser familiares. Este objeto clonado es solo una tabla de Snowflake estándar, que admite todas las operaciones de las tablas estándar a pesar del método de creación. 


In [ ]:
SELECT * 
FROM vegetable_details_clone;

## Snowflake Marketplace 📓

### ¿Qué es Snowflake Marketplace?

Snowflake Marketplace es un portal integrado al que puede acceder dentro de su cuenta de Snowflake. Puede usar Snowflake Marketplace para descubrir y acceder a datos y servicios de terceros, y comercializar sus propios productos de datos en la AI Data Cloud de Snowflake. Como consumidor, podría usar los datos proporcionados en Snowflake Marketplace para explorar y acceder a lo siguiente:

- Datos históricos para investigación, pronósticos y machine learning.
- Datos de streaming actualizados, como condiciones climáticas y del tráfico actuales.
- Datos de identidad especializados para comprender a los suscriptores y objetivos de audiencia.
- Nueva información de fuentes de datos imprevistas. 

### ¿Qué es más fácil que cargar datos en una tabla de Snowflake? Bueno, NO CARGARLOS en absoluto.

Snowflake Marketplace le permite realizar búsquedas en un rango de categorías de datos y productos de datos significativos para su negocio de un creciente grupo de proveedores globales y aprovecharlos dentro de su cuenta con tan solo hacer clic en un botón. Hay cientos de listas, incluidas opciones **Free/Unlimited Access**. Siempre que tenga los privilegios para realizar transacciones de Marketplace, los datos compartidos pueden estar disponibles en su cuenta en segundos y aparecer como una nueva base de datos. Pueden analizarse y utilizarse allí, e incluso unirse con sus propios datos, para aumentar, mejorar y crear nuevos productos de datos. ¡No se necesita cargar datos!

--- 

![Snowflake Marketplace (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_marketplace_v2.png)


### Desplácese hasta Snowflake Marketplace 🥋 

1. En el menú principal del lado izquierdo, seleccione **Data Products** > **Marketplace**.
1. Use la barra de búsqueda de Snowflake Marketplace > escriba “books” y pulse Enter.
1. En el submenú que aparece, seleccione **Pricing** > **Free**.
1. Busque la lista **AI Training Dataset from Goodreads Books**.
1. Haga clic en el enlace y revise los detalles proporcionados para este conjunto de datos, que incluye una especificación del diccionario de datos y ejemplos de uso (consultas SQL).

💡 **CONSEJO**: Las listas de Snowflake Marketplace requieren privilegios especiales para adquirir una cuenta. Por lo general, un administrador de datos o del sistema manejaría esto con acceso a roles de privilegios elevados en Snowflake.


## Acerca del intercambio privado con Data Exchange 📓

[Data Exchange](https://docs.snowflake.com/en/user-guide/data-exchange) proporciona un centro de datos para colaborar de forma segura en torno a los datos con un grupo seleccionado de miembros que usted invite. Como proveedor, le permite publicar datos, que luego los consumidores que participan en su intercambio pueden descubrir.

Con Data Exchange, puede proporcionar datos de manera sencilla a un grupo específico de partners de negocios consistentes que participan en Data Exchange, como departamentos internos de su compañía o proveedores, distribuidores y partners externos a su compañía. Supongamos que quiere compartir datos con distintos consumidores dentro y fuera de su organización. En ese caso, también puede usar las listas ofrecidas a consumidores específicos o de manera pública en Snowflake Marketplace.


### Trabajar con intercambio privado 🥋 

Además de Snowflake Marketplace “público”, también es posible compartir datos y productos de datos con una lista de cuentas de Snowflake seleccionadas o incluso con usuarios en una cuenta. Esta función se llamada **Data Exchange**.

1. En el menú principal del lado izquierdo, seleccione **Horizon Catalog** > **Data sharing** > **External sharing** > **Shared With You**.
1. Revise las listas compartidas con nuestra cuenta, muchas de Servicios de Educación de Snowflake.
1. Haga clic en la lista para **Alpine Peaks Publishing** y revísela.

Escribiremos consultas SQL en función de este conjunto de datos de nuestra cuenta de Snowflake para abrir las principales recomendaciones de libros para nuestro tema favorito: **jardinería**. ¡Siempre hay más para aprender!

💡 **NOTA**: Los Servicios de educación de Snowflake han adquirido ya este conjunto de datos de intercambio privado en nuestra cuenta. Usted **NO** necesita hacer clic en el botón azul **Get** ni en el enlace **Get Access**.

--- 

![Snowflake Data Exchange (imagen)](https://edu-cdev-images.s3.us-west-2.amazonaws.com/ob/ob_pde_alpine_peaks_listing_1_v2.png)

- Para que su rol pueda acceder a estos datos compartidos, ejecute la siguiente instrucción SQL, que le otorgará los permisos adecuados.

- Actualice su navegador de objetos y ahora debería poder ver la base de datos `ALPINE_PEAKS_PUBLISHING` que aparece.


In [ ]:
CALL common_db.resources.access_alpine_peaks_publishing_data('{{user}}_learner_rl');

### Consultar datos de Data Exchange privados 🥋

Aparece la siguiente descripción para la lista **Alpine Peaks Publishing** en Data Exchange:

- _Alpine Peaks Publishing, una editorial especializada en la publicación de libros de ficción, reveló su catálogo para el 2025 a Snowflake Education. La colección incluye diversos títulos de varios géneros, atractivos para una amplia audiencia. Con las obras de sus aclamados autores (pero ficticios), esta selección promete cautivar a los lectores con obras para todos los gustos._

Ejecute las consultas en las siguientes celdas SQL para explorar este conjunto de datos.


In [ ]:
-- ¿Cuántas reseñas de libros hay en este conjunto de datos?
SELECT COUNT(*) 
FROM alpine_peaks_publishing.books.forthcoming_releases;

In [ ]:
-- eche un vistazo a un subconjunto de estos datos
SELECT * 
FROM alpine_peaks_publishing.books.forthcoming_releases
LIMIT 10;

In [ ]:
-- determine el número de categorías únicas en este conjunto de datos
SELECT DISTINCT category 
FROM alpine_peaks_publishing.books.forthcoming_releases
ORDER BY 1;

In [ ]:
-- produzca un recuento del número total de libros por categoría
SELECT category, COUNT(*) 
FROM alpine_peaks_publishing.books.forthcoming_releases
GROUP BY category 
ORDER BY 2 DESC;

In [ ]:
-- encuentre libros con 'jardín' en su nombre
SELECT *
FROM alpine_peaks_publishing.books.forthcoming_releases
WHERE LOWER(title) LIKE '%garden%';

In [ ]:
-- liste los libros de nuestra categoría favorita 'jardinería' y ordénelos por la fecha en que estarán disponibles en 2025
SELECT *
FROM alpine_peaks_publishing.books.forthcoming_releases
WHERE CONTAINS(LOWER(category),'garden')
ORDER BY release_date;


## Conclusión de la Lección 6 🏁 

### ¿Todo listo para terminar la Lección 6? 🏁 

- ¿La tabla **ROOT_DEPTH** tiene tres filas? 

- ¿La tabla **VEGETABLE_DETAILS** tiene 41 filas?

- ¿Las tablas están en el esquema **VEGGIES** de la base de datos **(animal)_GARDEN_PLANTS**? 

Si la respuesta es SÍ a todas estas preguntas, debería marcar esta lección como completa. De lo contrario, debería volver y corregir lo que no sea correcto.


## Ponga a prueba sus conocimientos :mag_right:

Compruebe sus conocimientos con las preguntas interactivas que aparecen a continuación. Cada celda `RUN_THIS_QUIZ_QUESTION_` tiene un widget Streamlit que presenta una pregunta de opción múltiple sobre la funcionalidad de Snowflake.  

**Instrucciones:**  
1. Pase el cursor sobre la celda del cuaderno para revelar controles adicionales.
1. Haga clic en el botón ▶️ **Reproducir** en el lado derecho de cada celda del cuestionario para ejecutarla.  
1. Seleccione su respuesta entre las opciones que se proporcionan.  
1. Revise los comentarios antes de continuar. 

💡 **Nota:** Puede expandir la celda para ver el código si tiene curiosidad, pero no es obligatorio. Estos cuestionarios no son obligatorios: están disponibles para desafiar su transferencia de conocimientos y proporcionarle la oportunidad de practicar.  


In [ ]:
st.divider()
question = "¿Qué es un formato de archivo?"
options = ["Seleccione una opción a continuación...",
           "A) Es una forma de decirle a Snowflake cómo estará estructurada su datos cuando lleguen", 
           "B) Es una forma de establecer márgenes y fuente para una tabla", 
           "C) Es una forma de elegir el color de fondo de una página web"]
           
user_answer = st.radio(question, options, index=0)
if user_answer:
    if user_answer == "Seleccione una opción a continuación...": # esta opción es una solución alternativa hasta que Streamlit se actualice a > 1.26.0 para que podamos usar index=None
        ''
    else:
        answer = 'fc8e32c542c9f10af4af7a6bf654cf0c'
        # Extraer la letra de la opción (A, B, C o D)
        selected_option = user_answer.split(')')[0] + ')'
        response = session.sql(f"call common_db.resources.quiz_temp('{answer}', '{user_answer}', 'False')").collect()
        if response:
            value = response[0]['QUIZ_TEMP']
        st.write(f"{selected_option} {value}")

## Siguientes pasos

Si completó los pasos del laboratorio y respondió correctamente las preguntas de la **Prueba de conocimientos**, continúe al siguiente cuaderno cuando se lo indique el instructor de Snowflake.
